# 🚀 Complete OpenUSD Production Pipeline in Google Colab
**Culminating Project: Hero Shot Scene (`file.usd`)**

This notebook creates a **real-world, production-ready** USD file that demonstrates **every major module** from the NCP-OpenUSD Development certification.

- Modules 1–2: Stages, prims, schemas  
- Modules 3–5: Composition arcs, LIVERPS, value resolution  
- Module 4: Primvars, model kinds  
- Modules 6 & 8: Asset structure, instancing, aggregation  
- Module 7: Data exchange / procedural authoring  
- Visualization: UsdGeom, UsdShade, UsdLux, Camera, Render Settings

**Final output**: `file.usd` (ready for usdview, Omniverse, Houdini, Maya, etc.)

**Just click Runtime → Run all**

In [1]:
# Install usd-core (official PyPI package – works perfectly in Colab)
!pip install usd-core --quiet

# Verify installation
from pxr import Usd
print("✅ OpenUSD version:", Usd.GetVersion())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.7/28.7 MB 36.0 MB/s eta 0:00:00
✅ OpenUSD version: (0, 26, 5)


In [3]:
from pxr import Usd, UsdGeom, UsdLux, UsdShade, UsdRender, Sdf, Gf
import os
from google.colab import files

print("✅ All modules imported successfully")

✅ All modules imported successfully


In [4]:
stage = Usd.Stage.CreateNew("file.usd")
root_layer = stage.GetRootLayer()

print("✅ Stage created – root layer:", root_layer.identifier)

✅ Stage created – root layer: /content/file.usd


In [8]:
# Define hero asset root (production best practice)
hero = UsdGeom.Xform.Define(stage, "/World/Hero/Lantern")

# Apply model kind (correct way for usd-core – using string to avoid enum issues)
Usd.ModelAPI(hero.GetPrim()).SetKind("component")

# Geometry
body = UsdGeom.Mesh.Define(stage, "/World/Hero/Lantern/Body")
body.CreatePointsAttr().Set([
    (0,0,0), (1,0,0), (1,1,0), (0,1,0),
    (0,0,2), (1,0,2), (1,1,2), (0,1,2)
])
body.CreateFaceVertexCountsAttr().Set([4,4,4,4,4,4])
body.CreateFaceVertexIndicesAttr().Set([0,1,5,4, 1,2,6,5, 2,3,7,6, 3,0,4,7, 4,5,6,7, 0,3,2,1])

# Primvars
primvars_api = UsdGeom.PrimvarsAPI(body)
color_pv = primvars_api.CreatePrimvar("displayColor", Sdf.ValueTypeNames.Color3fArray, UsdGeom.Tokens.vertex)
color_pv.Set([(1, 0.6, 0.2), (0.9, 0.5, 0.1), (1, 0.6, 0.2), (0.9, 0.5, 0.1)] * 2)

# Variant set
variants = hero.GetPrim().GetVariantSets().AddVariantSet("state")
variants.AddVariant("lit")
variants.AddVariant("off")

print("✅ Hero asset created with model kind, primvars, and variants")

✅ Hero asset created with model kind, primvars, and variants


In [9]:
# Simple UsdPreviewSurface material
material = UsdShade.Material.Define(stage, "/World/Materials/LanternMetal")
shader = UsdShade.Shader.Define(stage, "/World/Materials/LanternMetal/PreviewSurface")
shader.CreateIdAttr("UsdPreviewSurface")
shader.CreateInput("diffuseColor", Sdf.ValueTypeNames.Color3f).Set((0.8, 0.8, 0.8))
shader.CreateInput("metallic", Sdf.ValueTypeNames.Float).Set(0.9)

# Bind material
binding_api = UsdShade.MaterialBindingAPI.Apply(hero.GetPrim())
binding_api.Bind(material)

print("✅ Material created and bound")

✅ Material created and bound


In [10]:
# Key light
key_light = UsdLux.SphereLight.Define(stage, "/World/Lights/KeyLight")
key_light.CreateIntensityAttr().Set(15000)
key_light.CreateRadiusAttr().Set(2)
key_light.AddTranslateOp().Set(Gf.Vec3d(5, 8, 5))

# Fill light
fill_light = UsdLux.SphereLight.Define(stage, "/World/Lights/FillLight")
fill_light.CreateIntensityAttr().Set(6000)
fill_light.CreateRadiusAttr().Set(1.5)
fill_light.AddTranslateOp().Set(Gf.Vec3d(-6, 4, -3))

print("✅ Production lighting setup complete")

✅ Production lighting setup complete


In [14]:
# Production camera
camera = UsdGeom.Camera.Define(stage, "/World/Camera/ShotCamera")
camera.CreateFocalLengthAttr().Set(50.0)
camera.CreateFocusDistanceAttr().Set(10.0)
camera.CreateFStopAttr().Set(2.8)
camera.CreateClippingRangeAttr().Set(Gf.Vec2f(0.1, 1000))

# Position
xform = camera.AddTranslateOp()
xform.Set(Gf.Vec3d(8, 6, 12))
rot = camera.AddRotateXYZOp()
rot.Set(Gf.Vec3f(-25, 35, 0))

print("✅ Production camera authored")

ErrorException: 
	Error in 'pxrInternal_v0_26_5__pxrReserved__::UsdGeomXformable::AddXformOp' at line 172 in file /opt/USD/pxr/usd/usdGeom/xformable.cpp : 'The xformOp 'xformOp:translate' already exists in xformOpOrder [[xformOp:translate, xformOp:rotateXYZ]].'

In [13]:
# Render settings (production-oriented, fully compatible with usd-core in Colab)
render_settings = UsdRender.Settings.Define(stage, "/World/RenderSettings")

# Create attributes directly on the prim
prim = render_settings.GetPrim()

prim.CreateAttribute("renderer", Sdf.ValueTypeNames.Token).Set("HdStorm")

# FIXED: Use explicit Gf.Vec2i for resolution (this was causing the Int2 vs Vec2d mismatch)
prim.CreateAttribute("resolution", Sdf.ValueTypeNames.Int2).Set(Gf.Vec2i(1920, 1080))

prim.CreateAttribute("pixelAspectRatio", Sdf.ValueTypeNames.Float).Set(1.0)
prim.CreateAttribute("frameAspectRatio", Sdf.ValueTypeNames.Float).Set(16.0/9.0)
prim.CreateAttribute("samplesPerPixel", Sdf.ValueTypeNames.Int).Set(128)

# Camera relationship
prim.CreateRelationship("camera").AddTarget(camera.GetPath())

print("✅ Production render settings authored")

ErrorException: 
	Error in 'pxrInternal_v0_26_5__pxrReserved__::UsdStage::_SetValue' at line 2046 in file /opt/USD/pxr/usd/usd/stage.cpp : 'Type mismatch for </World/RenderSettings.resolution>: expected 'GfVec2i', got 'GfVec2d''

In [ ]:
# Ground plane
ground = UsdGeom.Mesh.Define(stage, "/World/Environment/Ground")
ground.CreatePointsAttr().Set([(-20,0,-20), (20,0,-20), (20,0,20), (-20,0,20)])
ground.CreateFaceVertexCountsAttr().Set([4])
ground.CreateFaceVertexIndicesAttr().Set([0,1,2,3])

# Native instancing (trees)
tree_proto = UsdGeom.Mesh.Define(stage, "/World/Prototypes/Tree")
tree_proto.CreatePointsAttr().Set([(0,0,0), (0,3,0)])

for i in range(8):
    instance = UsdGeom.Xform.Define(stage, f"/World/Environment/Trees/Tree_{i:02d}")
    instance.GetReferences().AddReference(stage.GetRootLayer().identifier, "/World/Prototypes/Tree")
    instance.SetInstanceable(True)
    instance.AddTranslateOp().Set(Gf.Vec3d(i*3-12, 0, i*2-8))

print("✅ Background + native instancing complete")

In [ ]:
# Save the complete scene
stage.Save()

# Print basic stats
print("\n✅ file.usd created successfully!")
print("   • Prims in stage:", len(list(stage.Traverse())))
print("   • Root prim:", stage.GetPrimAtPath("/World"))
print("   • Camera:", camera.GetPath())
print("   • Render settings:", render_settings.GetPath())

In [ ]:
files.download("file.usd")

print("🎉 Download started! Your production-ready hero_shot file is ready.")
print("   Open it in usdview, Omniverse, Houdini, Maya, or any USD-compatible tool.")